In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

# Each line contains one raw JSON event
lines = spark.read.text(f"{BRONZE}/clickstream")

print("Total raw lines:", lines.count())

# Inspect five raw JSON events
display(lines.limit(5))


Total raw lines: 135780


value
"{""event_id"": ""48ec4d5ba7b8d791"", ""session_id"": ""s-d647d2c971e5"", ""customer_id"": 52, ""event_type"": ""add_to_cart"", ""event_ts"": ""2026-07-02T07:55:16.000Z"", ""product_id"": 3, ""order_id"": null, ""device"": ""mobile"", ""user_agent"": ""Mozilla/5.0 (iPhone; iOS 17_5) Safari/605.1"", ""utm_source"": ""direct""}"
"{""event_id"": ""088b667e79606508"", ""session_id"": ""s-1710e2edfcb4"", ""customer_id"": null, ""event_type"": ""product_view"", ""event_ts"": ""2026-07-02T09:32:51.000Z"", ""product_id"": 37, ""order_id"": null, ""device"": ""desktop"", ""user_agent"": ""python-requests/2.31"", ""utm_source"": ""meta""}"
"{""event_id"": ""4d9506925bf7fe51"", ""session_id"": ""s-b638460904aa"", ""customer_id"": null, ""event_type"": ""product_view"", ""event_ts"": ""2026-07-02T16:42:35.000Z"", ""product_id"": 22, ""order_id"": null, ""device"": ""mobile"", ""user_agent"": ""Mozilla/5.0 (Linux; Android 14) Chrome/126.0 Mobile"", ""utm_source"": ""google""}"
"{""event_id"": ""47109f0fd946ba6d"", ""session_id"": ""s-0b076fb6a4cb"", ""customer_id"": 289, ""event_type"": ""purchase"", ""event_ts"": ""2026-07-02T21:12:30.000Z"", ""product_id"": null, ""order_id"": 8799, ""device"": ""desktop"", ""user_agent"": ""Mozilla/5.0 (Windows NT 10.0; Win64) Chrome/126.0"", ""utm_source"": ""google""}"
"{""event_id"": ""5987450ecad16398"", ""session_id"": ""s-9c0bd5f72851"", ""customer_id"": null, ""event_type"": ""product_view"", ""event_ts"": ""2026-07-03T07:12:48.000Z"", ""product_id"": 48, ""order_id"": null, ""device"": ""desktop"", ""user_agent"": ""Googlebot/2.1 (+http://www.google.com/bot.html)"", ""utm_source"": ""email""}"


In [0]:

# Define the expected JSON structure
SCHEMA = (
    "event_id STRING, "
    "session_id STRING, "
    "customer_id BIGINT, "
    "event_type STRING, "
    "event_ts STRING, "
    "product_id BIGINT, "
    "order_id BIGINT, "
    "device STRING, "
    "user_agent STRING, "
    "utm_source STRING"
)

# Parse each JSON string into a structured record
parsed = lines.withColumn(
    "e",
    F.from_json(F.col("value"), SCHEMA)
)

# Identify records that cannot be used
bad_lines = parsed.filter(
    F.col("e.event_id").isNull()
).select("value")

# Records with a recognized event ID
valid_lines = parsed.filter(
    F.col("e.event_id").isNotNull()
)

print("Total raw lines:", lines.count())
print("Quarantined lines:", bad_lines.count())
print("Initially parsed lines:", valid_lines.count())

# Save malformed / missing-event-ID records separately
bad_lines.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("atliq_dl.silver.web_events_quarantine")

# Inspect the structured records
display(valid_lines.select("e.*").limit(5))


Total raw lines: 135780
Quarantined lines: 426
Initially parsed lines: 135354


event_id,session_id,customer_id,event_type,event_ts,product_id,order_id,device,user_agent,utm_source
48ec4d5ba7b8d791,s-d647d2c971e5,52,add_to_cart,2026-07-02T07:55:16.000Z,3,null,mobile,Mozilla/5.0 (iPhone; iOS 17_5) Safari/605.1,direct
088b667e79606508,s-1710e2edfcb4,null,product_view,2026-07-02T09:32:51.000Z,37,null,desktop,python-requests/2.31,meta
4d9506925bf7fe51,s-b638460904aa,null,product_view,2026-07-02T16:42:35.000Z,22,null,mobile,Mozilla/5.0 (Linux; Android 14) Chrome/126.0 Mobile,google
47109f0fd946ba6d,s-0b076fb6a4cb,289,purchase,2026-07-02T21:12:30.000Z,null,8799,desktop,Mozilla/5.0 (Windows NT 10.0; Win64) Chrome/126.0,google
5987450ecad16398,s-9c0bd5f72851,null,product_view,2026-07-03T07:12:48.000Z,48,null,desktop,Googlebot/2.1 (+http://www.google.com/bot.html),email


In [0]:

from pyspark.sql import functions as F

# Helper for text normalization
def norm(col):
    return F.lower(
        F.regexp_replace(F.trim(col), r"\s+", " ")
    )

# Handle both timestamp formats
ts = (
    F.when(
        F.col("event_ts").rlike(r"^\d+$"),
        F.timestamp_millis(F.col("event_ts").cast("bigint"))
    )
    .otherwise(
        F.try_to_timestamp(
            F.col("event_ts"),
            F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX")
        )
    )
)

# Clean parsed events
ev = (
    valid_lines
    .select("e.*")
    .withColumn("event_ts", ts)
    .withColumn("event_type", norm(F.col("event_type")))
    .dropDuplicates(["event_id"])
    .filter(
        F.col("session_id").isNotNull() &
        (F.trim(F.col("session_id")) != "")
    )
)

print("Events after initial cleaning:", ev.count())

display(
    ev.groupBy("event_type")
    .count()
    .orderBy("event_type")
)


Events after initial cleaning: 132036


event_type,count
add_to_cart,11882
checkout,4442
page_view,51490
product_view,60776
purchase,3446


In [0]:

from pyspark.sql import functions as F

# 1. Identify all sessions containing bot activity
bots = (
    ev.filter(
        F.col("user_agent").rlike(
            r"(?i)(bot|crawler|spider|python-requests)"
        )
    )
    .select("session_id")
    .distinct()
)

# 2. Remove all events belonging to those sessions
events_no_bots = ev.join(
    bots,
    "session_id",
    "left_anti"
)

# 3. Get valid product IDs from Silver Products
known_products = (
    spark.table("atliq_dl.silver.products")
    .select(F.col("product_id").alias("_pid"))
    .distinct()
)

# 4. Exclude events referencing unknown products
events_clean = (
    events_no_bots
    .join(
        known_products,
        F.col("product_id") == F.col("_pid"),
        "left"
    )
    .filter(
        F.col("product_id").isNull() |
        F.col("_pid").isNotNull()
    )
    .drop("_pid")
)

# 5. Check final metrics before writing
print("Events before bot filtering:", ev.count())
print("Bot sessions identified:", bots.count())
print("Events after bot filtering:", events_no_bots.count())
print("Final clean events:", events_clean.count())
print(
    "Final unique sessions:",
    events_clean.select("session_id").distinct().count()
)
print(
    "Invalid timestamps:",
    events_clean.filter(F.col("event_ts").isNull()).count()
)


Events before bot filtering: 132036
Bot sessions identified: 1059
Events after bot filtering: 93622
Final clean events: 93436
Final unique sessions: 23193
Invalid timestamps: 0


In [0]:

TARGET = "atliq_dl.silver.web_events"

# Save cleaned events as a managed Delta table
events_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET)

print("Clickstream Silver table created!")

# Verify the persisted table
silver_events = spark.table(TARGET)

display(
    silver_events.agg(
        F.count("*").alias("silver_events"),
        F.countDistinct("event_id").alias("unique_event_ids"),
        F.countDistinct("session_id").alias("unique_sessions")
    )
)


Clickstream Silver table created!


silver_events,unique_event_ids,unique_sessions
93436,93436,23193


In [0]:
%run ./utils_dq

In [0]:
print(callable(log_dq))

True


In [0]:
from pyspark.sql import functions as F

# Count duplicates and missing sessions separately
valid_events = valid_lines.select("e.*")
deduped_events = valid_events.dropDuplicates(["event_id"])

duplicate_events = valid_events.count() - deduped_events.count()

missing_sessions = deduped_events.filter(
    F.col("session_id").isNull() |
    (F.trim(F.col("session_id")) == "")
).count()

# Record Clickstream DQ metrics
log_dq("web_events", "duplicate_event_ids_removed", duplicate_events)

log_dq("web_events", "missing_session_removed", missing_sessions)

log_dq("web_events", "bot_sessions_identified", bots.count())

log_dq(
    "web_events",
    "bot_events_removed",
    ev.count() - events_no_bots.count()
)

log_dq(
    "web_events",
    "unknown_product_events_removed",
    events_no_bots.count() - events_clean.count()
)

log_dq(
    "web_events",
    "invalid_timestamps_found",
    events_clean.filter(F.col("event_ts").isNull()).count()
)

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'web_events'")
    .orderBy("rule")
)

run_date,table_name,rule,rows_affected
2026-11-01,web_events,bot_events_removed,38414
2026-10-31,web_events,bot_events_removed,38414
2026-10-31,web_events,bot_sessions_identified,1059
2026-11-01,web_events,bot_sessions_identified,1059
2026-10-31,web_events,duplicate_event_ids_removed,2680
2026-11-01,web_events,duplicate_event_ids_removed,2680
2026-10-31,web_events,invalid_timestamps_found,0
2026-11-01,web_events,invalid_timestamps_found,0
2026-10-31,web_events,missing_or_unparseable_event_id,426
2026-10-31,web_events,missing_session_removed,638
